###Mount Google Drive in Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print("Mounted? ", os.path.ismount("/content/drive"))
print("MyDrive exists? ", os.path.isdir("/content/drive/MyDrive"))
# If both True, skip mounting and proceed to use:
DATA_PATH = "/content/drive/MyDrive/CUAD_v1.json"

Mounted?  True
MyDrive exists?  True


In [ ]:
from google.colab import drive

# Unmount if mounted (ignore errors if not)
drive.flush_and_unmount()
!fusermount -u /content/drive 2>/dev/null || true

# Remove any stray local files at the mountpoint
!rm -rf /content/drive
!mkdir -p /content/drive

# Remount
drive.mount('/content/drive')


Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
import os
print(os.path.isdir('/content/drive/MyDrive'))  # True => already mounted


True


###Cache Folder

In [ ]:
CACHE_DIR = "/content/drive/MyDrive/legal_summarization_cache"
!mkdir -p "$CACHE_DIR"


### Quick setup

In [ ]:
!pip install torch transformers nltk tqdm rouge-score bert-score scikit-learn

import nltk
nltk.download('punkt')


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.2 MB/s eta 0:00:00
  C

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

###Paths

In [ ]:
import os
PROJECT_DIR = "/content/legal_summ_colab"
CACHE_DIR   = "/content/drive/MyDrive/legal_summarization_cache"  # persists in Drive

DATA_PATH = "/content/drive/MyDrive/CUAD_v1.json"
CORPUS_PKL = "/content/drive/MyDrive/legal_summarization_cache/corpus.pkl"


os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
CORPUS_PKL  = f"{CACHE_DIR}/corpus.pkl"
CKPT_DIR    = f"{CACHE_DIR}/runs_longformer_ranker"
print("Project dir:", PROJECT_DIR)
print("Cache dir  :", CACHE_DIR)
print("CUAD JSON  :", DATA_PATH)

Project dir: /content/legal_summ_colab
Cache dir  : /content/drive/MyDrive/legal_summarization_cache
CUAD JSON  : /content/drive/MyDrive/CUAD_v1.json


###Build sentence‑labeled corpus

In [ ]:
# Prep: CUAD -> sentence labels (positives overlap CUAD answers)
import json, re, pickle
from nltk.tokenize import sent_tokenize
from tqdm import tqdm

#### Remove hyphenation across line breaks

In [ ]:
def clean_text(s):
    if not isinstance(s, str): return ""
    s = re.sub(r"-\n\s*", "", s)
    s = re.sub(r"\n{2,}", "\n\n", s)
    s = re.sub(r"[ \t]+", " ", s)
    return s.strip()

In [ ]:

def iter_paragraphs(cuad_path):
    with open(cuad_path) as f:
        raw = json.load(f)
    data = raw["data"] if "data" in raw else raw
    for doc in data:
        for p in doc.get("paragraphs", []):
            ctx = clean_text(p.get("context",""))
            spans = []
            present_clause_types = set()
            for qa in p.get("qas", []):
                if qa.get("is_impossible") is False:
                    present_clause_types.add(qa.get("question",""))
                    for ans in qa.get("answers", []):
                        a0 = ans["answer_start"]; a1 = a0 + len(ans["text"])
                        spans.append((a0, a1))
            yield ctx, sorted(spans), sorted(present_clause_types)

In [ ]:

def sentence_labels(text, pos_spans):
    sents = sent_tokenize(text)
    offsets, cur = [], 0
    for s in sents:
        i = text.find(s, cur)
        offsets.append((i, i+len(s)))
        cur = i + len(s)
    labels = []
    for (s0,s1) in offsets:
        lab = 0
        for (a0,a1) in pos_spans:
            if max(s0,a0) < min(s1,a1): lab = 1; break
        labels.append(lab)
    return sents, offsets, labels

In [ ]:
def build_corpus(cuad_path):
    corpus = []
    for ctx, spans, clause_types in tqdm(iter_paragraphs(cuad_path), desc="Preparing CUAD"):
        if not ctx: continue
        sents, soff, labs = sentence_labels(ctx, spans)
        if sents:
            corpus.append({
                "text": ctx,
                "sentences": sents,
                "sent_char_spans": soff,
                "labels": labs,
                "present_clause_types": clause_types
            })
    return corpus

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')  # required by newer NLTK versions


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
if os.path.exists(CORPUS_PKL):
    print("Using cached corpus:", CORPUS_PKL)
    with open(CORPUS_PKL, "rb") as f: corpus = pickle.load(f)
else:
    print("⏳ Building corpus from CUAD (first run only)…")
    corpus = build_corpus(DATA_PATH)
    with open(CORPUS_PKL, "wb") as f: pickle.dump(corpus, f)
    print(f"💾 Saved {len(corpus)} paragraph samples to {CORPUS_PKL}")

print("Samples in corpus:", len(corpus))

Using cached corpus: /content/drive/MyDrive/legal_summarization_cache/corpus.pkl
Samples in corpus: 510


###Train Legal‑Longformer extractive ranker

In [ ]:
import math, random, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm import tqdm

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
CKPT_DIR   = CKPT_DIR if 'CKPT_DIR' in globals() else "/content/drive/MyDrive/legal_summarization_cache/runs_longformer_ranker"
CORPUS_PKL = CORPUS_PKL if 'CORPUS_PKL' in globals() else "/content/drive/MyDrive/legal_summarization_cache/corpus.pkl"

In [ ]:
ENCODER = "allenai/longformer-base-4096"  # You can swap to your clause-classifier checkpoint path

In [ ]:

# Load corpus if not in RAM
if 'corpus' not in globals():
    with open(CORPUS_PKL, "rb") as f:
        corpus = pickle.load(f)
print("Samples in corpus:", len(corpus))

Samples in corpus: 510


In [ ]:
# ---- Chunk-level dataset to avoid OOM ----
MAX_SEQ = 3072  # use 3072 or 2048 if T4 is tight; 4096 if you have headroom
STRIDE  = 256

In [ ]:
# Reproducible split
rng = random.Random(42)
corpus = corpus[:]  # shallow copy
rng.shuffle(corpus)
split = int(0.9 * len(corpus))
train_c, dev_c = corpus[:split], corpus[split:]
print(f"Train: {len(train_c)}  Dev: {len(dev_c)}")

Train: 459  Dev: 51


In [ ]:
class CUADChunkDataset(Dataset):
    def __init__(self, examples, tokenizer, max_seq=4096, stride=256):
        self.tok = tokenizer
        self.max_seq = max_seq
        self.stride  = stride
        self.chunks  = []

        for item in examples:
            text = item["text"]
            enc  = self.tok(text, return_offsets_mapping=True, add_special_tokens=True, truncation=False)
            offsets = enc["offset_mapping"]; input_ids = enc["input_ids"]

            # char -> token spans per sentence
            sent_tok_spans = []
            for (c0,c1) in item["sent_char_spans"]:
                idxs = [ti for ti,(o0,o1) in enumerate(offsets) if not (o1<=c0 or o0>=c1)]
                if idxs: t0, t1 = idxs[0], idxs[-1]+1
                else:    t0, t1 = 0, 1
                sent_tok_spans.append((t0,t1))

            # sliding window into chunks
            N = len(input_ids)
            start = 0
            while start < N:
                end = min(start + self.max_seq, N)
                spans_in, labels_in = [], []
                for j,(t0,t1) in enumerate(sent_tok_spans):
                    if not (t1 <= start or t0 >= end):
                        spans_in.append((max(0,t0-start), min(end-start, t1-start)))
                        labels_in.append(item["labels"][j])
                if spans_in:
                    self.chunks.append({
                        "input_ids": input_ids[start:end],
                        "attention_mask": [1]*(end-start),
                        "sent_spans": spans_in,
                        "sent_labels": labels_in
                    })
                if end == N: break
                start = end - self.stride

    def __len__(self): return len(self.chunks)
    def __getitem__(self, idx): return self.chunks[idx]

def collate_chunk(batch):
    maxT = max(len(x["input_ids"]) for x in batch)
    pad_id = 0
    input_ids, attention_mask, sent_spans, sent_labels = [], [], [], []
    for x in batch:
        pad = maxT - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [pad_id]*pad)
        attention_mask.append(x["attention_mask"] + [0]*pad)
        sent_spans.append(x["sent_spans"])
        sent_labels.append(x["sent_labels"])
    return (torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(attention_mask, dtype=torch.long),
            sent_spans, sent_labels)


In [ ]:
# Model
class ExtractiveRanker(nn.Module):
    def __init__(self, encoder_name=ENCODER, hidden=768, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        self.proj = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )
    def forward(self, input_ids, attention_mask, sent_spans):
        H = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        logits_list = []
        for b, spans in enumerate(sent_spans):
            if not spans:
                logits_list.append(torch.empty(0, device=H.device)); continue
            vecs = [H[b, s:e, :].mean(0) if e>s else H[b, s, :] for (s,e) in spans]
            V = torch.stack(vecs, 0)
            logits_list.append(self.proj(V).squeeze(-1))
        return logits_list

def bce_masked_loss(logits_list, labels_list, pos_weight=3.0):
    if not logits_list: return torch.tensor(0.0, requires_grad=True)
    maxS = max(len(x) for x in logits_list)
    B = len(logits_list); device = logits_list[0].device
    logits = torch.full((B,maxS), 0.0, device=device)
    labels = torch.full((B,maxS), -1.0, device=device)
    for i,(lg,lbl) in enumerate(zip(logits_list, labels_list)):
        S = len(lg)
        if S == 0: continue
        logits[i,:S] = lg
        labels[i,:S] = torch.tensor(lbl, device=device, dtype=torch.float)
    mask = labels >= 0
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([3.5], device=device), reduction='none')
    loss = (loss_fn(logits, labels) * mask.float()).sum() / mask.float().sum().clamp_min(1.0)
    return loss

# Tokenizer and datasets
tok = AutoTokenizer.from_pretrained(ENCODER)

ds_tr = CUADChunkDataset(train_c, tok, max_seq=MAX_SEQ, stride=STRIDE)
ds_dev = CUADChunkDataset(dev_c,   tok, max_seq=MAX_SEQ, stride=STRIDE)
print(f"Train chunks: {len(ds_tr)}  Dev chunks: {len(ds_dev)}")

dl_tr = DataLoader(ds_tr, batch_size=1, shuffle=True,  collate_fn=collate_chunk)
dl_dev = DataLoader(ds_dev, batch_size=1, shuffle=False, collate_fn=collate_chunk)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ExtractiveRanker().to(device)


Train chunks: 1984  Dev chunks: 203


pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]

In [ ]:
# Big memory saver:
model.encoder.gradient_checkpointing_enable()
model.encoder.config.use_cache = False


#### Optimizer / scheduler

In [ ]:
enc_params  = list(model.encoder.parameters())
head_params = list(model.proj.parameters())
opt = torch.optim.AdamW(
    [{"params": enc_params,  "lr": 2e-5},
     {"params": head_params, "lr": 1e-4}],
    weight_decay=0.01
)
epochs = 3
grad_accum = 8
total_steps = epochs * math.ceil(len(dl_tr)/grad_accum)
sch = get_linear_schedule_with_warmup(opt, int(0.05*total_steps), total_steps)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
best_dev = float("inf")

/tmp/ipython-input-731990995.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


#### Train (skip if already saved)

In [ ]:
if os.path.exists(CKPT_DIR):
    print("Using cached model:", CKPT_DIR)
else:
    os.makedirs(CKPT_DIR, exist_ok=True)
    for ep in range(1, epochs+1):
        model.train(); opt.zero_grad(); running = 0.0
        for it, batch in enumerate(tqdm(dl_tr, desc=f"Epoch {ep}")):
            ids, mask, spans, labels = batch
            ids, mask = ids.to(device), mask.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                logits_list = model(ids, mask, spans)
                loss = bce_masked_loss(logits_list, labels, pos_weight=3.5) / grad_accum
            scaler.scale(loss).backward()
            running += loss.item()
            if (it+1) % grad_accum == 0:
                scaler.step(opt); scaler.update(); opt.zero_grad(); sch.step()

        # Dev loss
        model.eval()
        with torch.no_grad():
            dev_losses = []
            for batch in dl_dev:
                ids, mask, spans, labels = batch
                ids, mask = ids.to(device), mask.to(device)
                ll = model(ids, mask, spans)
                dev_losses.append(bce_masked_loss(ll, labels, pos_weight=3.5).item())
        dev = sum(dev_losses)/max(1,len(dev_losses))
        print(f"DEV loss: {dev:.4f}")

        if dev < best_dev:
            best_dev = dev
            model.save_pretrained(CKPT_DIR)
            tok.save_pretrained(CKPT_DIR)
            print("Saved to", CKPT_DIR)

Using cached model: /content/drive/MyDrive/legal_summarization_cache/runs_longformer_ranker


###Evaluation Model

In [ ]:
import os, pickle, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer

device = "cuda" if torch.cuda.is_available() else "cpu"

###Load corpus

In [ ]:
with open(CORPUS_PKL, "rb") as f:
    corpus = pickle.load(f)
print("Corpus samples:", len(corpus))

# Simple 90/10 split (same as train cell; change seed if needed)
rng = np.random.RandomState(42)
idx = rng.permutation(len(corpus))
split = int(0.9*len(corpus))
dev_items = [corpus[i] for i in idx[split:]]
print("Dev samples:", len(dev_items))


Corpus samples: 510
Dev samples: 51


In [ ]:
import os
from transformers import AutoTokenizer

# 1) Make sure CKPT_DIR is set to your saved model folder
CKPT_DIR = "/content/drive/MyDrive/legal_summarization_cache/runs_longformer_ranker"

# 2) If tokenizer files are missing, write them into CKPT_DIR
need_save = not os.path.isdir(CKPT_DIR) or not any(
    fn in os.listdir(CKPT_DIR) for fn in ["tokenizer.json","vocab.json","merges.txt"]
)
if need_save:
    print("Tokenizer files missing in CKPT_DIR. Saving from base model…")
    base_name = "zlucia/legal-longformer-base-4096"   # or your encoder name
    tok_base = AutoTokenizer.from_pretrained(base_name, use_fast=True)
    os.makedirs(CKPT_DIR, exist_ok=True)
    tok_base.save_pretrained(CKPT_DIR)

# 3) Now load tokenizer from the checkpoint folder (works for eval/inference)
tok = AutoTokenizer.from_pretrained(CKPT_DIR, use_fast=True)

# sanity:
print("Tokenizer loaded. Files:", [f for f in os.listdir(CKPT_DIR) if "tokenizer" in f or f in ("vocab.json","merges.txt")])


Tokenizer files missing in CKPT_DIR. Saving from base model…


OSError: zlucia/legal-longformer-base-4096 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

###Utilities

In [ ]:
MAX_SEQ = 3072  # keep consistent with training
STRIDE  = 256
tok = AutoTokenizer.from_pretrained(CKPT_DIR)

TypeError: expected str, bytes or os.PathLike object, not NoneType